In [8]:
# Indicate Data Paths (where CHAT data is stored) and Output Paths (where you want parquets uploaded)
# Change only this cell, and everything should run

data_path = "../data/raw"
output_path = "../data/processed"

# CHAT Feature Engineering Pipeline
This notebook implements the CHATFeatureEngineer class, a robust automated pipeline designed to transform raw CHAT (.cha) transcripts from the CHILDES database into structured, analysis-ready Parquet files.

## Key Features
* Exhaustive Metadata Extraction: Captures session-level details across 21 standard CHAT headers (e.g., date, location, situation) and speaker-level metadata (e.g., age in months, role).
  
* Linguistic Metrics: Automatically computes standardized developmental measures using PyLangAcq, including MLU (Mean Length of Utterance in words and morphemes), TTR (Type-Token Ratio), and IPSyn (Index of Productive Syntax).
  
* Granular NLP Tiers: Uses regex-based parsing to extract and aggregate morphological information (%mor), grammatical relations (%gra), and phonological data (%xpho) for every utterance in a session.
  
* Conversational Context: Tracks the immediate temporal context by capturing the preceding and following utterances for every speaker turn.
  
* Hierarchical Processing: Recursively mirrors the raw data directory structure, exporting individual experiment files and generating a concatenated meta-file (e.g., _Eng-NA_extracted.parquet) for each language/study group.
  
* Data Integrity: Implements sophisticated logging and safety checks (_get_tier_safe, try-except blocks) to ensure high-throughput processing even when encountering inconsistent or legacy transcript formats.

In [9]:
import re
import pandas as pd
import numpy as np
import pylangacq
from pathlib import Path
from typing import List, Dict, Any, Tuple
import logging
from collections import defaultdict

logging.basicConfig(level=logging.INFO, format='%(asctime)s - [%(levelname)s] - %(message)s')
logger = logging.getLogger(__name__)

class CHATFeatureEngineer:
    def __init__(self, data_root: str):
        self.data_root = Path(data_root)
        self.target_headers = [
            'comments', 'date', 'languages', 'location', 'media', 
            'number', 'options', 'other', 'participants', 'pid', 
            'recording_quality', 'room_layout', 'situation', 
            'tape_location', 'time_duration', 'time_start', 
            'transcriber', 'transcription', 'types', 'videos', 'warning'
        ]

    def _get_canonical_id(self, session_id: str) -> str:
        """Strips trailing lowercase letters (a, b, c) to create a Super-Session ID."""
        return re.sub(r'[a-z]$', '', session_id)

    def _extract_provenance(self, file_path: Path) -> Dict[str, str]:
        parts = file_path.parts
        try:
            session_id = file_path.stem
            return {
                'language_id': parts[-4],
                'experiment_name': parts[-3],
                'subject_id': parts[-2],
                'session_id': session_id,
                'canonical_id': self._get_canonical_id(session_id)
            }
        except IndexError:
            return {k: 'unknown' for k in ['language_id', 'experiment_name', 'subject_id', 'session_id', 'canonical_id']}

    def _calculate_conversational_features(self, utterances: List[Any]) -> List[Dict]:
        features = []
        def get_text(u):
            if u and hasattr(u, 'tokens') and u.tokens:
                return " ".join([t.word for t in u.tokens if hasattr(t, 'word')])
            return None

        for i, utt in enumerate(utterances):
            time_marks = getattr(utt, 'time_marks', None)
            start, end = (time_marks[0], time_marks[1]) if time_marks and len(time_marks) == 2 else (np.nan, np.nan)
            
            features.append({
                'speaker': utt.participant,
                'start_time': start,
                'end_time': end,
                'duration': end - start if pd.notna(start) else np.nan,
                'prev_context': get_text(utterances[i-1]) if i > 0 else None,
                'next_context': get_text(utterances[i+1]) if i < len(utterances) - 1 else None
            })
        return features

    def _get_tier_safe(self, utt_obj, tier_name):
        if utt_obj and hasattr(utt_obj, 'tiers'):
            val = utt_obj.tiers.get(tier_name, np.nan)
            return val if val != "" else np.nan
        return np.nan

    def process_corpus(self, output_base: str):
        output_path = Path(output_base)
        pos_regex, mor_regex = r'(\w+)(?=\|)', r'(?<=\|)([^\s]+)'
        gra_rel_regex, gra_dep_regex = r'\|([^|\s]+)(?=\s|$)', r'(\d+\|\d+)(?=\|)'

        language_aggregates = {}
        processed_dirs = set()

        for cha_file in self.data_root.rglob("*.cha"):
            experiment_dir = cha_file.parent
            if experiment_dir in processed_dirs: continue
            processed_dirs.add(experiment_dir)

            relative_path = experiment_dir.parent.relative_to(self.data_root)
            lang_out_dir = output_path / relative_path
            lang_out_dir.mkdir(parents=True, exist_ok=True)

            logger.info(f"Ingesting Experiment: {experiment_dir.name}")
            try:
                reader = pylangacq.read_chat(str(experiment_dir))
            except Exception as e:
                logger.error(f"Failed to read {experiment_dir}: {e}"); continue

            # --- AGGREGATION LOGIC ---
            file_groups = defaultdict(list)
            for f_path in sorted(reader.file_paths):
                prov = self._extract_provenance(Path(f_path))
                group_key = (prov['subject_id'], prov['canonical_id'])
                file_groups[group_key].append(f_path)

            session_data = []
            all_headers = reader.headers()
            file_to_header = {path: all_headers[i] for i, path in enumerate(reader.file_paths)}

            for (subj, canon_id), paths in file_groups.items():
                super_reader = reader.filter(files=paths)
                all_super_utts = super_reader.utterances()
                
                # Pre-map roles for this super-session
                # We extract roles from the first available file in the group
                first_file_idx = reader.file_paths.index(paths[0])
                participants_obj = reader.participants(by_file=True)[first_file_idx]
                code_to_role = {p.code: (p.role if p.role else np.nan) for p in participants_obj}

                for speaker in code_to_role.keys():
                    try:
                        spk_reader = super_reader.filter(participants=[speaker])
                        spk_utts = spk_reader.utterances()
                        if not spk_utts: continue

                        # 1. Metrics Extraction (Old-style Robustness)
                        mlu = spk_reader.mlu(participant=speaker)[0] if spk_reader.mlu(participant=speaker) else np.nan
                        mluw = spk_reader.mluw(participant=speaker)[0] if spk_reader.mluw(participant=speaker) else np.nan
                        mlum = spk_reader.mlum(participant=speaker)[0] if spk_reader.mlum(participant=speaker) else np.nan
                        ttr = spk_reader.ttr(participant=speaker)[0] if spk_reader.ttr(participant=speaker) else np.nan

                        speaker_role = code_to_role.get(speaker, np.nan)

                        try:
                            ipsyn = spk_reader.ipsyn(participant=speaker)[0] if speaker_role == 'Target_Child' else np.nan
                        except:
                            ipsyn = np.nan

                        # 2. Age Extraction (Restored from Working Old Code)
                        ages_list = spk_reader.ages()
                        age = ages_list[0] if (ages_list and ages_list[0] is not None) else None
                        if age and hasattr(age, 'years'):
                            age_months = age.years * 12 + age.months + age.days / 30
                        else:
                            age_months = np.nan

                        # 3. Tier Extraction
                        raw_text, mor_full, gra_full, xpho, pos, mor_clean, g_rel, g_dep = [], [], [], [], [], [], [], []
                        
                        for utt in spk_utts:
                            u_raw = self._get_tier_safe(utt, speaker)
                            u_mor = self._get_tier_safe(utt, '%mor')
                            u_gra = self._get_tier_safe(utt, '%gra')
                            
                            raw_text.append(u_raw)
                            mor_full.append(u_mor)
                            gra_full.append(u_gra)
                            xpho.append(self._get_tier_safe(utt, '%xpho'))
                            pos.append(" ".join(re.findall(pos_regex, str(u_mor))) if pd.notna(u_mor) else np.nan)
                            mor_clean.append(" ".join(re.findall(mor_regex, str(u_mor))) if pd.notna(u_mor) else np.nan)
                            g_rel.append(" ".join(re.findall(gra_rel_regex, str(u_gra))) if pd.notna(u_gra) else np.nan)
                            g_dep.append(" ".join(re.findall(gra_dep_regex, str(u_gra))) if pd.notna(u_gra) else np.nan)

                        # 4. Context Extraction
                        temporal = self._calculate_conversational_features(all_super_utts)
                        spk_temporal = [f for f in temporal if f['speaker'] == speaker]

                        # 5. Build Row
                        prov_base = self._extract_provenance(Path(paths[0]))
                        row = {
                            'language_id': prov_base['language_id'],
                            'experiment_name': prov_base['experiment_name'],
                            'subject_id': subj,
                            'session_id': canon_id, # Canonical ID as the session identifier
                            'speaker_code': speaker,
                            'speaker_role': speaker_role,
                            'speaker_age_months': age_months,
                            'mlu_score': mlu,
                            'mluw_score': mluw,
                            'mlum_score': mlum,
                            'ttr_score': ttr,
                            'ipsyn_score': ipsyn,
                            'n_utterances': len(spk_utts),
                            'text_part_of_speech': str(pos),
                            'text_morphological_info': str(mor_clean),
                            'text_grammatical_relation_type': str(g_rel),
                            'text_grammatical_dependency': str(g_dep),
                            'text_phonological_info': str(xpho),
                            'text_raw': str(raw_text),
                            'context_previous_utterance': str([f['prev_context'] for f in spk_temporal]),
                            'context_next_utterance': str([f['next_context'] for f in spk_temporal])
                        }

                        # 6. Metadata Handling (Merge Strategy)
                        for h_key in self.target_headers:
                            # Use the header from the first part of the session
                            val = getattr(file_to_header[paths[0]], h_key, np.nan)
                            if isinstance(val, (list, dict, set)):
                                row[f"meta_{h_key}"] = str(val) if val else np.nan
                            else:
                                row[f"meta_{h_key}"] = val if val not in ["", None] else np.nan

                        session_data.append(row)
                    except Exception as e:
                        logger.warning(f"Error in speaker {speaker} @ {canon_id}: {e}")

            # Save individual data file
            if session_data:

                # Convert to DataFrame
                df = pd.DataFrame(session_data)
                df.replace(['', 'None', 'nan', 'NaN', '', '{}'], np.nan, inplace=True)
                
                # Export to Parquet
                output_file = lang_out_dir / f"{experiment_dir.name}_extracted.parquet"
                df.to_parquet(output_file)
                logger.info(f"Successfully exported {len(df)} records to {output_file}")

                # TRACK FOR METAFILE: Add df to its language group
                if relative_path not in language_aggregates:
                    language_aggregates[relative_path] = []
                language_aggregates[relative_path].append(df)

        # --- AFTER ALL LOOPS: CREATE METAFILES ---
        for lang_rel_path, df_list in language_aggregates.items():

            if df_list:

                # Concatenate all experiments for this language
                meta_df = pd.concat(df_list, ignore_index=True)
                
                # The filename: prefix with underscore and save in the parent folder
                meta_filename = f"_{lang_rel_path.name}_extracted.parquet"
                meta_output_path = output_path / lang_rel_path / meta_filename
                
                meta_df.to_parquet(meta_output_path)
                logger.info(f"Created METAFILE: {meta_output_path} with {len(meta_df)} total records.")

        return meta_df

In [10]:
# Run CHAT data engineer class
engineer = CHATFeatureEngineer(data_root=data_path)
df = engineer.process_corpus(output_base=output_path)
display(df.head())

2026-03-19 01:30:24,145 - [INFO] - Ingesting Experiment: Eve
2026-03-19 01:30:25,521 - [INFO] - Successfully exported 47 records to ../data/processed/Eng-NA/Brown/Eve_extracted.parquet
2026-03-19 01:30:25,521 - [INFO] - Ingesting Experiment: Adam
2026-03-19 01:30:29,190 - [INFO] - Successfully exported 248 records to ../data/processed/Eng-NA/Brown/Adam_extracted.parquet
2026-03-19 01:30:29,191 - [INFO] - Ingesting Experiment: Sarah
2026-03-19 01:30:32,958 - [INFO] - Successfully exported 545 records to ../data/processed/Eng-NA/Brown/Sarah_extracted.parquet
2026-03-19 01:30:33,001 - [INFO] - Created METAFILE: ../data/processed/Eng-NA/Brown/_Brown_extracted.parquet with 840 total records.


,language_id,experiment_name,subject_id,session_id,speaker_code,speaker_role,speaker_age_months,mlu_score,mluw_score,mlum_score,ttr_score,ipsyn_score,n_utterances,text_part_of_speech,text_morphological_info,text_grammatical_relation_type,text_grammatical_dependency,text_phonological_info,text_raw,context_previous_utterance,context_next_utterance,meta_comments,meta_date,meta_languages,meta_location,meta_media,meta_number,meta_options,meta_other,meta_participants,meta_pid,meta_recording_quality,meta_room_layout,meta_situation,meta_tape_location,meta_time_duration,meta_time_start,meta_transcriber,meta_transcription,meta_types,meta_videos,meta_warning
0,Eng-NA,Brown,Eve,010600,CHI,Target_Child,18.0,1.430000,1.500000,1.430000,0.245714,14.0,1214,"['adj noun', 'adj noun', 'adj noun', 'propn', ...","['more-Cmp-S1 cookie', 'more-Cmp-S1 cookie', '...","['AMOD ROOT PUNCT', 'AMOD ROOT PUNCT', 'AMOD R...","['1|2 2|0 3|2', '1|2 2|0 3|2', '1|2 2|0 3|2', ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","['xxx more cookie .', 'xxx more cookie .', 'mo...","[None, 'here you go .', 'you have another cook...","['you more cookies ?', 'you have another cooki...",NaN,15-OCT-1962,['eng'],NaN,NaN,NaN,NaN,NaN,"[Participant(code='CHI', name='', role='Target...",11312/c-00034743-1,NaN,NaN,NaN,NaN,10:00-11:00,NaN,NaN,NaN,"long, toyplay, TD",NaN,NaN
1,Eng-NA,Brown,Eve,010600,MOT,Mother,NaN,4.330000,4.320000,4.330000,0.371429,NaN,1445,"['pron adj noun', 'intj det noun noun', 'aux p...","['you-Prs-Acc-S2 more-Cmp-S1 cookie-Plur', 'ho...","['NSUBJ AMOD ROOT PUNCT', 'DISCOURSE DET COMPO...","['1|3 2|3 3|0 4|3', '1|4 2|4 3|4 4|0 5|4', '1|...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","['you 0v more cookies ?', 'how_about another g...","['more cookie .', 'you more cookies ?', 'how_a...","['how_about another graham cracker ?', 'would ...",NaN,15-OCT-1962,['eng'],NaN,NaN,NaN,NaN,NaN,"[Participant(code='CHI', name='', role='Target...",11312/c-00034743-1,NaN,NaN,NaN,NaN,10:00-11:00,NaN,NaN,NaN,"long, toyplay, TD",NaN,NaN
2,Eng-NA,Brown,Eve,010600,COL,Investigator,NaN,3.900000,3.900000,3.900000,0.504274,NaN,138,"['pron pron noun verb', 'aux pron verb pron', ...","['what-Int-S1 that-Dem your do-Part-Pres-S', '...","['ROOT NSUBJ NSUBJ ACL-RELCL PUNCT', 'AUX NSUB...","['1|0 2|4 3|4 4|1 5|1', '1|3 2|3 3|0 4|3 5|3',...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[""what's that you're doing ?"", 'can you do it ...","['.', 'block broke .', 'can you do it ?', 'the...","['Mommy .', 'can you do it like that ?', 'ther...",NaN,15-OCT-1962,['eng'],NaN,NaN,NaN,NaN,NaN,"[Participant(code='CHI', name='', role='Target...",11312/c-00034743-1,NaN,NaN,NaN,NaN,10:00-11:00,NaN,NaN,NaN,"long, toyplay, TD",NaN,NaN
3,Eng-NA,Brown,Eve,010600,RIC,Investigator,NaN,3.615385,3.615385,3.615385,0.702128,NaN,23,"['pron aux pron verb', 'pron', 'pron adv verb ...",['what-Int-S1 be-Fin-Ind-Pres-S2 you-Prs-Nom-S...,"['OBJ AUX NSUBJ ROOT PUNCT', 'ROOT PUNCT', 'NS...","['1|4 2|4 3|4 4|0 5|4', '1|0 2|1', '1|3 2|3 3|...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","['what are you doing ?', 'what ?', ""I don('t) ...","['give dolly her bottle .', 'oh Fraser hat .',...","['water .', 'oh Fraser hat .', 'what_about the...",NaN,15-OCT-1962,['eng'],NaN,NaN,NaN,NaN,NaN,"[Participant(code='CHI', name='', role='Target...",11312/c-00034743-1,NaN,NaN,NaN,NaN,10:00-11:00,NaN,NaN,NaN,"long, toyplay, TD",NaN,NaN
4,Eng-NA,Brown,Eve,010700,CHI,Target_Child,19.0,2.140000,2.160000,2.140000,0.342857,16.0,838,"['adj noun', 'adj noun noun adv', 'intj', 'int...","['more-Cmp-S1 coffee', 'more-Cmp-S1 grape juic...","['AMOD ROOT PUNCT', 'AMOD COMPOUND ROOT ADVMOD...","['1|2 2|0 3|2', '1|3 2|3 3|0 4|3 5|3', '1|0 2|...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","['more coffee .', 'more grape juice too . [+ R...","[None, 'more coffee ?', 'Eve !', 'no .', ""yes ...","['more coffee ?', ""that's better ."", ""that's ...",NaN,12-NOV-1962,['eng'],NaN,NaN,NaN,NaN,NaN,"[Participant(code='CHI', name

# CHAT Extra Engineering Pipeline

### Feature Documentation: CHILDES Multi-Track Refinement Output
This document outlines the features engineered by the CHILDESRefiner class. The refiner transforms raw CHILDES Parquet data (native list format) into three distinct analysis tracks: Structural, Lexical (Clean), and Behavioral, alongside session-level Environmental metrics.

---

### 1. Structural Sequence Tracks (NLP-Ready)
These features convert nested linguistic metadata into space-separated string sequences suitable for sequence modeling (RNNs, Transformers) or N-gram analysis.
* pos_sequence: A space-separated string of Part-of-Speech tags (e.g., "v n det"). Extracted from the prefix of morphological tags.
* morph_sequence: A space-separated string of full morphological information (e.g., "v|look prep|at det|the n|dog").
* gra_rel_sequence: A space-separated sequence of grammatical relation types (e.g., "SUBJ OBJ JCT").
* gra_dep_sequence: A space-separated sequence of grammatical dependency indices.
* phonological_sequence: A space-separated string of phonological transcriptions (if available in the source).
* pos_variety_score: A count of unique POS tags within the utterance, serving as a proxy for syntactic complexity.
---
### 2. Lexical & Behavioral Tracks
Derived from the text_raw column by applying CHAT-specific regex resolution.
* text_clean: The "pure" lexical track. It resolves replacements (e.g., [: dog] becomes dog), removes all CHAT coding, markers, and brackets. Lowercased and stripped for standard NLP tasks like word embeddings or sentiment analysis.
* text_behavioral: A "hybrid" track for modeling communicative behavior. It preserves the word order but injects explicit tokens for CHAT events (e.g., [ERROR], [PAUSE], [REP], [REPLACE]).
---
### 3. Quantitative Behavioral Metrics (Utterance-Level)
Integer counts of specific CHAT markers detected within each utterance.
* n_unintelligible: Count of xxx, yyy, or www markers.
* n_omissions: Count of omitted words (marked with 0).
* n_errors: Count of developmental or speech errors (marked with [* ]).
* n_fragments/n_fillers: Counts of word fragments (&+) and filled pauses (&-).
* n_pauses: Count of silent pauses (e.g., (.) or (2.3)).
* n_repetitions/n_retracings/n_reformulations: Counts of disfluency markers [/], [//], and [///].
* n_overlaps: Count of overlapping speech instances.
* n_imitations: Binary-style count of the [+ IMIT] flag.
* n_responses: Binary-style count of the [+ RES] flag.
---
### 4. Interaction & Environmental Features
Calculated by aggregating the "Input" (Adult/Other speakers) and comparing it to the Target Child.
* imitation_score: A Jaccard-style overlap coefficient between the Child's current utterance and the previous speaker's utterance. Measures lexical priming or echolalia.
* env_adult_mlu: The mean Mean Length of Utterance of all non-child speakers in the session.
* env_adult_utterances: Total count of utterances produced by the environment.
* env_adult_text_clean: A concatenated string of all adult speech in the session for global lexical analysis.
* env_adult_pos_sequence: A concatenated sequence of all adult POS tags to analyze the syntactic environment.
* env_roles_present: A unique list of speaker roles (e.g., "Mother, Father, Investigator") present in the session.
* adult_to_child_utt_ratio: A ratio of adult-to-child verbal productivity.
---
### 5. Data Integrity
* Original Columns: All original columns (including text_morphological_info_full, context_next_utterance, and the native text_raw lists) are preserved in the final output to allow for custom back-referencing.

In [11]:
import pandas as pd
import numpy as np
import ast
import re
from typing import List, Dict, Any
from pathlib import Path
import logging

pd.set_option('display.max_columns', None)

logging.basicConfig(level=logging.INFO, format='%(asctime)s - [%(levelname)s] - %(message)s')
logger = logging.getLogger(__name__)

class CHILDESRefiner:
    """
    Expert-level refiner for CHILDES data. 
    Transforms native list columns into multidimensional, ML/NLP-ready units.
    Preserves all original data while adding structural and behavioral tracks.
    """
    
    def __init__(self):
        self.patterns = {
            # Meta-cleaning patterns
            'post_codes': r'\+\s?[a-zA-Z<:]{1,10}',  
            'comments': r'=[^ ]+',                   
            'structural_symbols': r'[<>/^]',          # Angles and slashes
            
            # Existing patterns
            'replacement': r'(?:\<[^>]+\>|\S+)\s+\[:\s+([^\]]+)\]', 
            'error': r'\[\*[^\]]*\]',
            'unintelligible': r'xxx|yyy|www',         # Removed \b for better matching
            'omission': r'\b0[a-zA-Z_0-9]*\b',
            'fragment': r'&\+[a-zA-Z0-9_]+',
            'filler': r'&-[a-zA-Z0-9_]+',
            'simple_event': r'&=\S+',
            'scoped_event': r'\[=!?[^\]]+\]|\[%[^\]]+\]',
            'pause': r'\(\.+|\(\d+(?::\d+)?\.\d*\)',
            'repetition': r'\[/\]',
            'retracing': r'\[//\]',
            'reformulation': r'\[///\]',
            'false_start': r'\[/-\]',
            'blocking': r'≠',
            'overlap': r'\[[><]\d*\]|\+<',
            'terminator_trailoff': r'\+\.\.\.\??',
            'terminator_interrupt': r'\+/\.\??',
            'terminator_self_interrupt': r'\+//\.\??',
            'imitation_flag': r'\[\+ IMIT\]',
            'response_flag': r'\[\+ RES\]',
            'emphasis_flag': r'\[!!\]',
            'all_brackets': r'\[.*?\]', 
            'word_markers': r'[@&]\w+',    # Updated to catch & fragments too
            'internal_symbols': r'[:·]', 
            'scoping_angles': r'[<>]',  
            'whitespace': r'\s+'
        }

        self.meta_cols = [
            'meta_comments', 'meta_date', 'meta_lang', 'meta_loc', 'meta_media', 'meta_num', 
            'meta_opt', 'meta_misc', 'meta_parts', 'meta_pid', 'meta_rec_qual', 'meta_room', 
            'meta_sit', 'meta_tape_loc', 'meta_duration', 'meta_time_start', 'meta_staff', 
            'meta_trans_type', 'meta_types', 'meta_videos', 'meta_warning'
        ]

    def _safe_parse(self, val: Any) -> List[str]:
        """Safely handles lists in strings"""

        # If it's a list with one giant string, grab the string
        if isinstance(val, list) and len(val) == 1:
            val = val[0]
            
        if isinstance(val, str):
            # Remove brackets and quotes
            clean_str = re.sub(r"[\[\],']", "", val)
            # Split by the list separator to get individual items
            return [item.strip() for item in clean_str.split(',')]
        
        return val

    def _engineer_structural_sequences(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Dynamically generates sequences, variety scores, and unique sets for all structural columns.
        Includes logic to zero out 'nan' sequences.
        """

        # {Original_Column: Target_Prefix}
        struct_map = {
            'text_morphological_info': 'morph',
            'text_part_of_speech': 'pos',
            'text_grammatical_relation_type': 'gra_rel',
            'text_grammatical_dependency': 'gra_dep',
            'text_phonological_info': 'pho'
        }

        for col, prefix in struct_map.items():
            if col in df.columns:
                # 1. Generate the base sequence string
                df[f'text_{prefix}'] = df[col].apply(lambda x: self._safe_parse(x)[0])
                
                # 2. Extract tokens (handling the 'nan' string explicitly)
                # We filter out 'nan' strings during the split to ensure they aren't counted
                def get_clean_tokens(seq_str):
                    tokens = [t for t in str(seq_str).split() if t.lower() != 'nan']
                    return tokens

                # 3. Generate Metrics
                df[f'{prefix}_tokens'] = df[f'text_{prefix}'].apply(get_clean_tokens)
                df[f'{prefix}_variety_score'] = df[f'{prefix}_tokens'].apply(len)
                df[f'{prefix}_sequence_unique'] = df[f'{prefix}_tokens'].apply(
                    lambda x: " ".join(sorted(list(set(x)))) if x else ""
                )
                
                # Clean up the helper token column
                df.drop(columns=[f'{prefix}_tokens'], inplace=True)
                
                # Final sanity check: if the sequence itself was just 'nan', clear the text column too
                df.loc[df[f'text_{prefix}'].str.lower() == 'nan', f'text_{prefix}'] = ""

        return df

    def _engineer_text_tracks(self, text_list: str) -> Dict[str, Any]:
        full_raw = text_list
        
        # 1. Metric Counts
        metrics = {
            'n_unintelligible': len(re.findall(self.patterns['unintelligible'], full_raw)),
            'n_omissions': len(re.findall(self.patterns['omission'], full_raw)),
            'n_errors': len(re.findall(self.patterns['error'], full_raw)),
            'n_fragments': len(re.findall(self.patterns['fragment'], full_raw)),
            'n_fillers': len(re.findall(self.patterns['filler'], full_raw)),
            'n_pauses': len(re.findall(self.patterns['pause'], full_raw)),
            'n_events': len(re.findall(self.patterns['simple_event'], full_raw)) + len(re.findall(self.patterns['scoped_event'], full_raw)),
            'n_repetitions': len(re.findall(self.patterns['repetition'], full_raw)),
            'n_retracings': len(re.findall(self.patterns['retracing'], full_raw)),
            'n_reformulations': len(re.findall(self.patterns['reformulation'], full_raw)),
            'n_interruptions': len(re.findall(self.patterns['terminator_interrupt'], full_raw)),
            'n_overlaps': len(re.findall(self.patterns['overlap'], full_raw)),
            'n_imitations': len(re.findall(self.patterns['imitation_flag'], full_raw)),
            'n_responses': len(re.findall(self.patterns['response_flag'], full_raw)),
        }

        # Pre-clean metadata
        base_text = re.sub(self.patterns['post_codes'], '', full_raw)
        base_text = re.sub(self.patterns['comments'], '', base_text)

        # --- 2. Track 1: text_clean_terminators (Level 1: All Punctuation) ---
        cl_text = re.sub(self.patterns['replacement'], r'\1', base_text)
        
        # Remove CHAT codes but preserve natural punctuation/symbols
        for key in ['unintelligible', 'omission', 'fragment', 'filler', 'simple_event', 
                    'pause', 'structural_symbols', 'all_brackets', 'word_markers', 'scoping_angles']:
            cl_text = re.sub(self.patterns[key], ' ', cl_text)
        
        # Normalize terminators
        cl_text = re.sub(self.patterns['terminator_trailoff'], '...', cl_text)
        cl_text = re.sub(self.patterns['terminator_interrupt'], '.', cl_text)
        
        text_full = re.sub(self.patterns['whitespace'], ' ', cl_text).strip() #.lower()
        metrics['text_clean_terminators'] = text_full

        # --- Track 1b: text_clean_terminators_impt (Level 2: Only . ? !) ---
        # Strip everything except letters, numbers, spaces, and the Big Three
        text_impt = re.sub(r'[^a-zA-Z0-9\s.?!\']', ' ', text_full)
        metrics['text_clean_terminators_impt'] = re.sub(self.patterns['whitespace'], ' ', text_impt).strip()

        # --- Track 1c: text_clean (Level 3: No Punctuation) ---
        # Strip all remaining punctuation
        text_none = re.sub(r'[.?!\',:;()\-]', ' ', text_impt)
        metrics['text_clean'] = re.sub(self.patterns['whitespace'], ' ', text_none).strip()

        # --- 3. Track 2: text_behavioral ---
        bh_text = re.sub(self.patterns['replacement'], r'\1 TOKEN_REPLACE', base_text)

        substitutions = {
            'error': ' TOKEN_ERROR ', 'unintelligible': ' TOKEN_UNINTEL ', 
            'omission': ' TOKEN_OMISSION ', 'fragment': ' TOKEN_FRAG ', 
            'filler': ' TOKEN_FILLER ', 'simple_event': ' TOKEN_EVENT ',
            'scoped_event': ' TOKEN_COMMENT ', 'pause': ' TOKEN_PAUSE ', 
            'repetition': ' TOKEN_REP ', 'structural_symbols': ' TOKEN_REP ',
            'retracing': ' TOKEN_RETR ', 'reformulation': ' TOKEN_REFORM ', 
            'false_start': ' TOKEN_FALSESTART ', 'overlap': ' TOKEN_OVERLAP '
        }
        for key, token in substitutions.items():
            bh_text = re.sub(self.patterns[key], token, bh_text)

        bh_text = re.sub(self.patterns['all_brackets'], ' ', bh_text)
        bh_text = re.sub(self.patterns['word_markers'], ' ', bh_text)
        bh_text = re.sub(self.patterns['internal_symbols'], ' ', bh_text)
        bh_text = re.sub(self.patterns['scoping_angles'], ' ', bh_text)
        bh_text = re.sub(self.patterns['structural_symbols'], ' ', bh_text)

        bh_text = re.sub(r'TOKEN_(\w+)', r'[\1]', bh_text)
        metrics['text_behavioral'] = re.sub(self.patterns['whitespace'], ' ', bh_text).strip()

        return metrics

    def _calculate_imitation(self, child_utts: List[str], prev_utts: List[str]) -> float:
        """Calculates set-based intersection to estimate imitation/echolalia."""

        if not child_utts or not prev_utts: 
            return 0.0
        
        c_set = set([str(x).lower() for x in child_utts if pd.notna(x)])
        p_set = set([str(x).lower() for x in prev_utts if pd.notna(x)])

        if not p_set: 
            return 0.0
        
        return len(c_set.intersection(p_set)) / len(p_set)

    def process(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        
        # 1. Standardize List Columns
        list_cols = ['text_raw', 'context_previous_utterance', 'context_next_utterance']
        for col in list_cols:
            if col in df.columns:
                df[col] = df[col].apply(lambda x: self._safe_parse(x)[0])

        # 2. Structural and Lexical Tracks
        logger.info("Engineering structural sequences and text tracks...")
        df = self._engineer_structural_sequences(df)
        chat_features = df['text_raw'].apply(self._engineer_text_tracks)
        chat_df = pd.DataFrame(chat_features.tolist(), index=df.index)
        df = pd.concat([df, chat_df], axis=1)

        # 3. Separate Child and Environment (Adults/Peers)
        child_df = df[df['speaker_role'].fillna('') == 'Target_Child'].copy()
        other_df = df[df['speaker_role'].fillna('') != 'Target_Child'].copy()

        # 4. Imitation Feature Calculation
        child_df['imitation_score'] = child_df.apply(
            lambda row: self._calculate_imitation(row['text_raw'], row['context_previous_utterance']), axis=1
        )

        # 5. Aggregate Adult Environment
        logger.info("Aggregating Adult linguistic environment metrics...")
        agg_dict = {
            'env_adult_mlu': pd.NamedAgg(column='mlu_score', aggfunc='mean'),
            'env_adult_utterances': pd.NamedAgg(column='n_utterances', aggfunc='sum'),
            'env_roles_present': pd.NamedAgg(column='speaker_role', aggfunc=lambda x: ", ".join(set(x))),
            'env_adult_text_clean': pd.NamedAgg(column='text_clean', aggfunc=lambda x: " ".join(x.astype(str))),
            'env_adult_text_behavioral': pd.NamedAgg(column='text_behavioral', aggfunc=lambda x: " ".join(x.astype(str))),
            'env_adult_errors': pd.NamedAgg(column='n_errors', aggfunc='sum'),
            'env_adult_pauses': pd.NamedAgg(column='n_pauses', aggfunc='sum'),
            'env_adult_retracings': pd.NamedAgg(column='n_retracings', aggfunc='sum')
        }
        
        if 'pos_variety_score' in other_df.columns:
            agg_dict['env_adult_pos_variety'] = pd.NamedAgg(column='pos_variety_score', aggfunc='mean')
        if 'pos_sequence' in other_df.columns:
            agg_dict['env_adult_pos_sequence'] = pd.NamedAgg(column='pos_sequence', aggfunc=lambda x: " ".join(x.astype(str)))

        env_agg = other_df.groupby('session_id').agg(**agg_dict).reset_index()

        # 6. Final Merge
        logger.info("Preparing final data...")
        final_df = pd.merge(child_df, env_agg, on='session_id', how='left')
        
        # Interaction Metric
        final_df['adult_to_child_utt_ratio'] = (
            final_df['env_adult_utterances'] / final_df['n_utterances']
        ).replace([np.inf, -np.inf], 0).fillna(0)

        # Defining Columns
        rename_map = {

            # Identifiers & Metadata
            'language_id': 'lang',
            'experiment_name': 'exp',
            'subject_id': 'subj_id',
            'session_filename': 'file',
            'session_id': 'sess_id',
            'speaker_code': 'spk',
            'speaker_role': 'role',
            'speaker_age_months': 'age_mo',
            'meta_comments': 'meta_comments',
            'meta_date': 'meta_date',
            'meta_languages': 'meta_lang',
            'meta_location': 'meta_loc',
            'meta_media': 'meta_media',
            'meta_number': 'meta_num',
            'meta_options': 'meta_opt',
            'meta_other': 'meta_misc',
            'meta_participants': 'meta_parts',
            'meta_pid': 'meta_pid',
            'meta_recording_quality': 'meta_rec_qual',
            'meta_room_layout': 'meta_room',
            'meta_situation': 'meta_sit',
            'meta_tape_location': 'meta_tape_loc',
            'meta_time_duration': 'meta_duration',
            'meta_time_start': 'meta_time_start',
            'meta_transcriber': 'meta_staff',
            'meta_transcription': 'meta_trans_type',
            'meta_types': 'meta_types',
            'meta_videos': 'meta_videos',
            'meta_warning': 'meta_warning',

            # Core Linguistic Scores
            'mlu_score': 'mlu',
            'mluw_score': 'mlu_w',
            'mlum_score': 'mlu_m',
            'ttr_score': 'ttr',
            'ipsyn_score': 'ipsyn',
            'n_utterances': 'n_utt',
            'imitation_score': 'imit_idx',
            'adult_to_child_utt_ratio': 'env_ratio',

            # Raw / Context Text
            'text_raw': 'raw',
            'text_clean': 'clean',
            'text_clean_terminators': 'clean_punct',
            'text_clean_terminators_impt': 'clean_impt',
            'text_behavioral': 'behav',
            'context_previous_utterance': 'prev_utt',
            'context_next_utterance': 'next_utt',

            # Structural: Part of Speech (POS)
            'text_part_of_speech': 'pos_raw',
            'text_pos': 'pos_seq',
            'pos_variety_score': 'pos_var',
            'pos_sequence_unique': 'pos_uniq',

            # Structural: Morphology
            'text_morphological_info': 'morph_raw',
            'text_morph': 'morph_seq',
            'text_morphological_info_full': 'morph_full',
            'morph_variety_score': 'morph_var',
            'morph_sequence_unique': 'morph_uniq',

            # Structural: Grammar Relations
            'text_grammatical_relation_type': 'gra_raw',
            'text_gra_rel': 'gra_seq',
            'text_grammatical_relation_full': 'gra_full',
            'gra_rel_variety_score': 'gra_var',
            'gra_rel_sequence_unique': 'gra_uniq',

            # Structural: Dependencies
            'text_grammatical_dependency': 'dep_raw',
            'text_gra_dep': 'dep_seq',
            'gra_dep_variety_score': 'dep_var',
            'gra_dep_sequence_unique': 'dep_uniq',

            # Structural: Phonology
            'text_phonological_info': 'pho_raw',
            'text_pho': 'pho_seq',
            'pho_variety_score': 'pho_var',
            'pho_sequence_unique': 'pho_uniq',

            # Behavioral Counts (Disfluencies/Events)
            'n_unintelligible': 'n_unint',
            'n_omissions': 'n_omiss',
            'n_errors': 'n_err',
            'n_fragments': 'n_frag',
            'n_fillers': 'n_fill',
            'n_pauses': 'n_pause',
            'n_events': 'n_ev',
            'n_repetitions': 'n_rep',
            'n_retracings': 'n_ret',
            'n_reformulations': 'n_ref',
            'n_interruptions': 'n_int',
            'n_overlaps': 'n_ov',
            'n_imitations': 'n_imit',
            'n_responses': 'n_res',

            # Environmental (Adult/Environment) Aggregates
            'env_adult_mlu': 'env_mlu',
            'env_adult_utterances': 'env_n_utt',
            'env_roles_present': 'env_roles',
            'env_adult_text_clean': 'env_clean',
            'env_adult_text_behavioral': 'env_behav',
            'env_adult_errors': 'env_err',
            'env_adult_pauses': 'env_pause',
            'env_adult_retracings': 'env_ret',
            'env_adult_pos_variety': 'env_pos_var',

        }

        final_df.rename(columns=rename_map, inplace=True)

        final_cols = [
            'sess_id', 'subj_id', 'lang', 'exp', 'age_mo',                                                   # Identifiers
            'raw', 'clean', 'clean_impt', 'clean_punct', 'behav',                                            # Speaker Text/Context
            'mlu', 'mlu_w', 'mlu_m', 'ttr', 'ipsyn', 'n_utt',                                                # Speaker Scores
            'env_mlu', 'env_n_utt', 'env_ratio', 'prev_utt', 'next_utt', 'env_clean', 'env_behav',           # Environmental Text/Context + Scores
            'pos_seq', 'pos_var', 'pos_uniq',                                                                # Speaker Structural Tracks
            'morph_seq', 'morph_var', 'morph_uniq',
            'gra_seq', 'gra_var', 'gra_uniq',
            'dep_seq', 'dep_var', 'dep_uniq',
            'pho_seq', 'pho_var', 'pho_uniq',
            'n_unint', 'n_omiss', 'n_err', 'n_frag', 'n_fill', 'n_pause',                                    # Speaker Behavioral Counts
            'n_ev', 'n_rep', 'n_ret', 'n_ref', 'n_int', 'n_ov', 'n_imit', 'n_res'
        ] + self.meta_cols

        # Final Arrangement of Data with Metadata at End
        final_df = final_df[final_cols]
        cols = [c for c in final_df.columns if c not in self.meta_cols]
        final_df = final_df[cols + [c for c in self.meta_cols if c in final_df.columns]]

        logger.info("Done!")

        return final_df
    


class CHILDESRefineryManager:
    """
    Automated manager for the CHILDESRefiner.
    Navigates processed language directories to transform 'extracted' files 
    into 'final' ML-ready datasets.
    """
    
    def __init__(self, processed_root: str):
        self.processed_root = Path(processed_root)
        self.refiner = CHILDESRefiner()

    def run_refinery(self):
        """
        Walks through lang/exp folders, processes data, and saves to the _final structure.
        """
        if not self.processed_root.exists():
            logger.error(f"Root path not found: {self.processed_root}")
            return

        # 1. Iterate through Language folders (e.g., Eng-NA, Spanish, etc.)
        # We skip folders starting with underscores (like '_final')
        for lang_dir in self.processed_root.iterdir():
            if not lang_dir.is_dir() or lang_dir.name.startswith('_'):
                continue

            lang_name = lang_dir.name
            logger.info(f"--- Processing Language Group: {lang_name}")

            # 2. Iterate through Experiment folders (e.g., Brown, Rollins, etc.)
            for exp_dir in lang_dir.iterdir():
                # Only look into directories, skipping the special '_final' output folder
                if not exp_dir.is_dir() or exp_dir.name == '_final':
                    continue

                exp_name = exp_dir.name
                input_file = exp_dir / f"_{exp_name}_extracted.parquet"

                if not input_file.exists():
                    # This happens if an experiment folder exists but CHATFeatureEngineer hasn't run yet
                    logger.info("-- No extracted data found. Skipping...")
                    continue

                try:
                    logger.info(f"-- Found extracted data for {lang_name}/{exp_name}. Loading...")
                    df_extracted = pd.read_parquet(input_file)

                    # 4. Perform the Transformation
                    logger.info(f"-- Refining {exp_name} features...")
                    df_final = self.refiner.process(df_extracted)

                    # 5. Define Output Path: ../data/processed/{lang}/_final/{exp}.parquet
                    output_dir = lang_dir / "_final" 
                    output_dir.mkdir(parents=True, exist_ok=True)
                    output_file = output_dir / f"{exp_name}.parquet"

                    # 6. Save and Log
                    df_final.to_parquet(output_file)
                    logger.info(f"-- Successfully saved final dataset to: {output_file}")
                    logger.info(f"-- Experiment {exp_name} Complete. Shape: {df_final.shape}")

                except Exception as e:
                    logger.error(f"-- Failed to refine experiment {exp_name}: {e}", exc_info=True)


        logger.info("--- All Refinery Tasks Completed")
        logger.info(f"\n\nSample Data: \n")
        display(df_final.head(1))

        logger.info(f"\n\nSample Text: \n")
        print("Raw: " + df_final.head(1)[['raw']].iloc[0,0])
        print("Clean: " + df_final.head(1)[['clean']].iloc[0,0])
        print("Clean (main punct): " + df_final.head(1)[['clean_impt']].iloc[0,0])
        print("Clean (all punct): " + df_final.head(1)[['clean_punct']].iloc[0,0])
        print("Behav: " + df_final.head(1)[['behav']].iloc[0,0])
        print("Pos: " + df_final.head(1)[['pos_seq']].iloc[0,0])
        print("Morph: " + df_final.head(1)[['morph_seq']].iloc[0,0])
        print("Gra: " + df_final.head(1)[['gra_seq']].iloc[0,0])
        print("Dep: " + df_final.head(1)[['dep_seq']].iloc[0,0])
        print("Pho: " + df_final.head(1)[['pho_seq']].iloc[0,0])
        print("Env: " + df_final.head(1)[['env_clean']].iloc[0,0])

In [12]:
## Run CHILDESRefinerManager 
## -- preferred vs CHILDESRefiner() because it processes multiple files
## -- to run CHILDESRefiner, do refiner = CHILDESRefiner() then df = refiner.process(df), assuming df is one of the extracted parquet files from CHATFeatureEngineer

if __name__ == "__main__":
    manager = CHILDESRefineryManager(processed_root=output_path)
    manager.run_refinery()

2026-03-19 01:30:33,135 - [INFO] - --- Processing Language Group: Eng-NA


2026-03-19 01:30:33,145 - [INFO] - -- Found extracted data for Eng-NA/Brown. Loading...
2026-03-19 01:30:33,169 - [INFO] - -- Refining Brown features...
2026-03-19 01:30:33,278 - [INFO] - Engineering structural sequences and text tracks...
2026-03-19 01:30:35,006 - [INFO] - Aggregating Adult linguistic environment metrics...
2026-03-19 01:30:35,022 - [INFO] - Preparing final data...
2026-03-19 01:30:35,026 - [INFO] - Done!
2026-03-19 01:30:35,065 - [INFO] - -- Successfully saved final dataset to: ../data/processed/Eng-NA/_final/Brown.parquet
2026-03-19 01:30:35,065 - [INFO] - -- Experiment Brown Complete. Shape: (200, 73)
2026-03-19 01:30:35,065 - [INFO] - --- All Refinery Tasks Completed
2026-03-19 01:30:35,066 - [INFO] - 

Sample Data: 



,sess_id,subj_id,lang,exp,age_mo,raw,clean,clean_impt,clean_punct,behav,mlu,mlu_w,mlu_m,ttr,ipsyn,n_utt,env_mlu,env_n_utt,env_ratio,prev_utt,next_utt,env_clean,env_behav,pos_seq,pos_var,pos_uniq,morph_seq,morph_var,morph_uniq,gra_seq,gra_var,gra_uniq,dep_seq,dep_var,dep_uniq,pho_seq,pho_var,pho_uniq,n_unint,n_omiss,n_err,n_frag,n_fill,n_pause,n_ev,n_rep,n_ret,n_ref,n_int,n_ov,n_imit,n_res,meta_comments,meta_date,meta_lang,meta_loc,meta_media,meta_num,meta_opt,meta_misc,meta_parts,meta_pid,meta_rec_qual,meta_room,meta_sit,meta_tape_loc,meta_duration,meta_time_start,meta_staff,meta_trans_type,meta_types,meta_videos,meta_warning
0,010600,Eve,Eng-NA,Brown,18.0,xxx more cookie . xxx more cookie . more juice...,more cookie more cookie more juice Fraser Fras...,more cookie . more cookie . more juice ? Frase...,more cookie . more cookie . more juice ? Frase...,[UNINTEL] more cookie . [UNINTEL] more cookie ...,1.43,1.5,1.43,0.245714,14.0,1214,3.948462,1606,1.3229,None here you go . you have another cookie rig...,you more cookies ? you have another cookie rig...,you more cookies how about another graham crac...,you [OMISSION] more cookies ? how_about anothe...,adj noun adj noun adj noun propn propn propn p...,1799,adj adp adv aux c cconj cm det intj l noun num...,more-Cmp-S1 cookie more-Cmp-S1 cookie more-Cmp...,1799,Becky Clipclop Cromer Daddy Dumpty DumptyDumpt...,AMOD ROOT PUNCT AMOD ROOT PUNCT AMOD ROOT PUNC...,2945,ADVMOD AMOD AUX CASE CC COMPOUND COMPOUND-PRT ...,1|2 2|0 3|2 1|2 2|0 3|2 1|2 2|0 3|2 1|0 2|1 1|...,2945,10|6 11|6 1|0 1|2 1|3 1|4 2|0 2|1 2|3 2|4 3|0 ...,nan nan nan nan nan nan nan nan nan nan nan na...,0,,179,10,0,0,1,43,1,0,0,0,0,28,0,0,NaN,15-OCT-1962,['eng'],NaN,NaN,NaN,NaN,NaN,"[Participant(code='CHI', name='', role='Target...",11312/c-00034743-1,NaN,NaN,NaN,NaN,10:00-11:00,NaN,NaN,NaN,"long, toyplay, TD",NaN,NaN


2026-03-19 01:30:35,074 - [INFO] - 

Sample Text: 



Raw: xxx more cookie . xxx more cookie . more juice ? Fraser . + RES Fraser . + RES Fraser . + RES Fraser . + RES yeah . + RES xxx (th)at ? a fly =! gestures . fly . Mommy telephone . my telephone . xxx . + RES Mommy . no . man / man . xxx . + RES xxx more cookie . block broke . there !! . I did it . there . there (.) Fraser . xxx . + RES xxx . baby . Mommy read . a stool . Fraser . + RES Fraser . + RES xxx more cookie . xxx more cookie . <little little little> / little . + IMIT milk ? <milk milk> / milk . that ? Fraser water ? oh Fraser . bye . + RES +< xxx water . Fraser water . Fraser water . that ? coffee . Fraser coffee . down / down . + IMIT ^ whispering cookie . Mommy . Mommy . a fly =! gestures . read the puzzle . read the puzzle . read the puzzle . read the puzzle . <read the puzzle read the puzzle> / read the puzzle . Racketyboom . "hat % putting hat on Leos head ." xxx . + RES m:hm ? . +< xxx water . bottle . +< xxx water . there . Fraser hat . oh Fraser hat . <oh Fraser hat